# VEP / Waveform Preprocessing Notebook for Noisy Raw Traces

This notebook prepares raw `.txt` waveform files for later VEP or general waveform analysis.

It focuses on:

- loading raw traces from a folder
- extracting the signal column
- handling missing/invalid values
- baseline correction
- optional detrending
- smoothing / low-pass filtering
- artifact rejection
- signal normalization
- quality-control plots
- exporting cleaned traces and summary metrics

The goal is **not** to perform final N1/P1/P2 peak analysis here.  
The goal is to create a clean dataset that is ready for downstream waveform analysis.


## 1. Install / import libraries

Required packages:

```bash
pip install numpy pandas matplotlib scipy
```

Optional, if using JupyterLab:

```bash
pip install jupyterlab ipykernel
```


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, savgol_filter, medfilt
from scipy.stats import median_abs_deviation

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True


## 2. User settings

Edit these values for your experiment.

Important assumptions based on the original MATLAB workflow:

- each trace is stored in a `.txt` file
- the waveform signal is in **column 4** in MATLAB
- MATLAB column 4 corresponds to Python column index `3`
- each trace has approximately 494 samples

If your files have headers or delimiters, adjust `read_csv` settings in the loading function below.


In [ ]:
# Folder containing raw .txt waveform files
DATA_DIR = Path("data/raw_vep_txt")

# Output folder for cleaned data
OUTPUT_DIR = Path("outputs/preprocessed_waveforms")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# MATLAB column 4 = Python index 3
SIGNAL_COLUMN_INDEX = 3

# Expected number of samples per waveform.
# Use None to allow variable-length traces.
EXPECTED_LENGTH = 494

# Sampling rate in Hz.
# If your x-axis is literally 1 sample = 1 ms, use 1000 Hz.
FS = 1000

# Baseline region in samples.
# Example: first 50 ms/samples.
BASELINE_START = 0
BASELINE_END = 50

# Filtering settings
APPLY_MEDIAN_FILTER = True
MEDIAN_KERNEL_SIZE = 3  # must be odd

APPLY_SAVGOL_FILTER = True
SAVGOL_WINDOW = 11      # must be odd
SAVGOL_POLYORDER = 3

APPLY_LOWPASS_FILTER = False
LOWPASS_CUTOFF_HZ = 80

# Artifact rejection settings
ARTIFACT_Z_THRESHOLD = 8.0
MAX_ALLOWED_NAN_FRACTION = 0.05

print("Input folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


## 3. Helper functions

These functions handle:

- robust file loading
- baseline correction
- filtering
- artifact detection
- summary metrics

The filtering is intentionally conservative.  
For VEP traces, you usually want to reduce noise without distorting peak timing.


In [ ]:
def load_trace_txt(path, signal_col=3):
    """Load one .txt trace and return the selected signal column as a NumPy array.

    This tries a flexible read first using pandas with automatic delimiter detection.
    If your files have a fixed delimiter, you can replace sep=None with sep='\t' or sep=','.
    """
    try:
        df = pd.read_csv(path, sep=None, engine="python", header=None)
    except Exception:
        # fallback for whitespace-delimited files
        df = pd.read_csv(path, delim_whitespace=True, header=None)

    if signal_col >= df.shape[1]:
        raise ValueError(f"{path.name}: requested column {signal_col}, but file only has {df.shape[1]} columns")

    y = pd.to_numeric(df.iloc[:, signal_col], errors="coerce").to_numpy(dtype=float)
    return y


def interpolate_nans(y):
    """Linearly interpolate NaN values."""
    y = np.asarray(y, dtype=float)
    if not np.isnan(y).any():
        return y

    x = np.arange(len(y))
    good = np.isfinite(y)

    if good.sum() < 2:
        return y

    y_interp = y.copy()
    y_interp[~good] = np.interp(x[~good], x[good], y[good])
    return y_interp


def baseline_correct(y, start=0, end=50, method="median"):
    """Subtract baseline estimated from a pre-stimulus or early segment."""
    baseline_region = y[start:end]

    if method == "median":
        baseline_value = np.nanmedian(baseline_region)
    elif method == "mean":
        baseline_value = np.nanmean(baseline_region)
    else:
        raise ValueError("method must be 'median' or 'mean'")

    return y - baseline_value, baseline_value


def robust_zscore(y):
    """Robust z-score using median and MAD."""
    med = np.nanmedian(y)
    mad = median_abs_deviation(y, nan_policy="omit", scale="normal")

    if mad == 0 or np.isnan(mad):
        return np.zeros_like(y)

    return (y - med) / mad


def lowpass_filter(y, fs=1000, cutoff_hz=80, order=4):
    """Zero-phase Butterworth low-pass filter."""
    nyq = fs / 2
    norm_cutoff = cutoff_hz / nyq

    if norm_cutoff >= 1:
        raise ValueError("LOWPASS_CUTOFF_HZ must be less than Nyquist frequency, FS/2")

    b, a = butter(order, norm_cutoff, btype="low")
    return filtfilt(b, a, y)


def preprocess_trace(y):
    """Preprocess one waveform trace."""
    y_raw = np.asarray(y, dtype=float)

    nan_fraction = np.isnan(y_raw).mean()
    y_work = interpolate_nans(y_raw)

    # Baseline correction first
    y_basecorr, baseline_value = baseline_correct(
        y_work,
        BASELINE_START,
        BASELINE_END,
        method="median"
    )

    y_filtered = y_basecorr.copy()

    # Median filter removes isolated spike artifacts
    if APPLY_MEDIAN_FILTER:
        y_filtered = medfilt(y_filtered, kernel_size=MEDIAN_KERNEL_SIZE)

    # Savitzky-Golay smoothing preserves peak shape better than a moving average
    if APPLY_SAVGOL_FILTER:
        # window must be odd and less than trace length
        window = min(SAVGOL_WINDOW, len(y_filtered) - 1 if len(y_filtered) % 2 == 0 else len(y_filtered))
        if window % 2 == 0:
            window -= 1
        if window > SAVGOL_POLYORDER + 2:
            y_filtered = savgol_filter(y_filtered, window_length=window, polyorder=SAVGOL_POLYORDER)

    # Optional low-pass filter
    if APPLY_LOWPASS_FILTER:
        y_filtered = lowpass_filter(y_filtered, fs=FS, cutoff_hz=LOWPASS_CUTOFF_HZ)

    # Artifact score after baseline correction
    z = robust_zscore(y_basecorr)
    max_abs_z = np.nanmax(np.abs(z))

    reject = False
    reject_reasons = []

    if nan_fraction > MAX_ALLOWED_NAN_FRACTION:
        reject = True
        reject_reasons.append("too_many_nans")

    if max_abs_z > ARTIFACT_Z_THRESHOLD:
        reject = True
        reject_reasons.append("large_artifact_or_spike")

    if EXPECTED_LENGTH is not None and len(y_raw) != EXPECTED_LENGTH:
        reject = True
        reject_reasons.append("unexpected_length")

    metrics = {
        "n_samples": len(y_raw),
        "nan_fraction": nan_fraction,
        "baseline_value": baseline_value,
        "raw_min": np.nanmin(y_raw),
        "raw_max": np.nanmax(y_raw),
        "corrected_min": np.nanmin(y_basecorr),
        "corrected_max": np.nanmax(y_basecorr),
        "filtered_min": np.nanmin(y_filtered),
        "filtered_max": np.nanmax(y_filtered),
        "max_abs_robust_z": max_abs_z,
        "rejected": reject,
        "reject_reasons": ";".join(reject_reasons)
    }

    return y_raw, y_basecorr, y_filtered, metrics


## 4. Load and preprocess all `.txt` files

This cell processes every `.txt` file in the selected folder.

Outputs created:

- `preprocessing_summary.csv`
- `cleaned_waveforms.csv`
- `raw_waveforms.csv`
- `baseline_corrected_waveforms.csv`

Each row in the waveform CSVs is one file/trace.  
Each column after `file` is one time/sample point.


In [ ]:
txt_files = sorted(DATA_DIR.glob("*.txt"))

if len(txt_files) == 0:
    print("No .txt files found. Check DATA_DIR.")
else:
    print(f"Found {len(txt_files)} .txt files")

summary_rows = []
raw_rows = []
basecorr_rows = []
filtered_rows = []

for path in txt_files:
    try:
        y = load_trace_txt(path, signal_col=SIGNAL_COLUMN_INDEX)
        y_raw, y_basecorr, y_filtered, metrics = preprocess_trace(y)

        summary_rows.append({"file": path.name, **metrics})

        raw_rows.append({"file": path.name, **{f"t_{i}": v for i, v in enumerate(y_raw)}})
        basecorr_rows.append({"file": path.name, **{f"t_{i}": v for i, v in enumerate(y_basecorr)}})
        filtered_rows.append({"file": path.name, **{f"t_{i}": v for i, v in enumerate(y_filtered)}})

    except Exception as e:
        summary_rows.append({
            "file": path.name,
            "n_samples": np.nan,
            "nan_fraction": np.nan,
            "baseline_value": np.nan,
            "raw_min": np.nan,
            "raw_max": np.nan,
            "corrected_min": np.nan,
            "corrected_max": np.nan,
            "filtered_min": np.nan,
            "filtered_max": np.nan,
            "max_abs_robust_z": np.nan,
            "rejected": True,
            "reject_reasons": f"load_or_processing_error: {e}"
        })

summary_df = pd.DataFrame(summary_rows)
raw_df = pd.DataFrame(raw_rows)
basecorr_df = pd.DataFrame(basecorr_rows)
filtered_df = pd.DataFrame(filtered_rows)

summary_df.to_csv(OUTPUT_DIR / "preprocessing_summary.csv", index=False)
raw_df.to_csv(OUTPUT_DIR / "raw_waveforms.csv", index=False)
basecorr_df.to_csv(OUTPUT_DIR / "baseline_corrected_waveforms.csv", index=False)
filtered_df.to_csv(OUTPUT_DIR / "cleaned_waveforms.csv", index=False)

summary_df.head()


## 5. Quality-control summary

This summarizes how many traces were retained or rejected.


In [ ]:
if not summary_df.empty:
    display(summary_df["rejected"].value_counts(dropna=False).rename("count"))

    if "reject_reasons" in summary_df.columns:
        display(summary_df["reject_reasons"].value_counts().rename("count"))

    display(summary_df.describe(include="all"))


## 6. Plot examples: raw vs baseline-corrected vs filtered

This lets you visually check whether filtering is helping or distorting the waveform.

If the filtered trace changes peak timing too much, reduce smoothing strength.


In [ ]:
def row_to_trace(df, row_idx):
    row = df.iloc[row_idx]
    values = row.drop(labels=["file"]).to_numpy(dtype=float)
    return row["file"], values


n_examples = min(5, len(filtered_df))

for idx in range(n_examples):
    file_name, y_raw = row_to_trace(raw_df, idx)
    _, y_basecorr = row_to_trace(basecorr_df, idx)
    _, y_clean = row_to_trace(filtered_df, idx)

    x = np.arange(len(y_raw))

    plt.figure(figsize=(14, 4))
    plt.plot(x, y_raw, label="raw", alpha=0.5)
    plt.plot(x, y_basecorr, label="baseline corrected", alpha=0.8)
    plt.plot(x, y_clean, label="filtered / cleaned", linewidth=2)
    plt.title(file_name)
    plt.xlabel("sample / ms if 1 kHz")
    plt.ylabel("uV or signal units")
    plt.legend()
    plt.show()


## 7. Plot all retained cleaned traces plus average ± SEM

This is the main QC plot before waveform analysis.


In [ ]:
if not filtered_df.empty and not summary_df.empty:
    retained_files = summary_df.loc[summary_df["rejected"] == False, "file"].tolist()
    retained_df = filtered_df[filtered_df["file"].isin(retained_files)].copy()

    if len(retained_df) == 0:
        print("No retained traces. Relax artifact rejection settings or inspect input files.")
    else:
        trace_matrix = retained_df.drop(columns=["file"]).to_numpy(dtype=float)
        mean_trace = np.nanmean(trace_matrix, axis=0)
        sem_trace = np.nanstd(trace_matrix, axis=0) / np.sqrt(trace_matrix.shape[0])
        x = np.arange(trace_matrix.shape[1])

        plt.figure(figsize=(14, 5))
        for row in trace_matrix:
            plt.plot(x, row, color="black", alpha=0.15, linewidth=0.7)

        plt.plot(x, mean_trace, color="red", linewidth=2.5, label="Mean cleaned trace")
        plt.fill_between(x, mean_trace - sem_trace, mean_trace + sem_trace, color="red", alpha=0.2, label="SEM")

        plt.title(f"Retained cleaned traces: n={trace_matrix.shape[0]}")
        plt.xlabel("sample / ms if 1 kHz")
        plt.ylabel("baseline-corrected signal")
        plt.legend()
        plt.show()

        avg_df = pd.DataFrame({
            "sample": x,
            "mean": mean_trace,
            "sem": sem_trace,
            "mean_minus_sem": mean_trace - sem_trace,
            "mean_plus_sem": mean_trace + sem_trace
        })
        avg_df.to_csv(OUTPUT_DIR / "average_cleaned_trace_sem.csv", index=False)
        display(avg_df.head())


## 8. Optional: inspect rejected traces

This is useful for deciding whether rejection criteria are too strict or too permissive.


In [ ]:
rejected_files = summary_df.loc[summary_df["rejected"] == True, "file"].tolist() if not summary_df.empty else []

print(f"Rejected traces: {len(rejected_files)}")

for file_name in rejected_files[:10]:
    if file_name in raw_df["file"].values:
        idx = raw_df.index[raw_df["file"] == file_name][0]
        _, y_raw = row_to_trace(raw_df, idx)
        _, y_clean = row_to_trace(filtered_df, idx)

        reason = summary_df.loc[summary_df["file"] == file_name, "reject_reasons"].iloc[0]

        plt.figure(figsize=(14, 4))
        plt.plot(y_raw, label="raw", alpha=0.6)
        plt.plot(y_clean, label="filtered", linewidth=2)
        plt.title(f"{file_name} | rejected: {reason}")
        plt.xlabel("sample / ms if 1 kHz")
        plt.ylabel("signal")
        plt.legend()
        plt.show()


## 9. Export retained traces only

This creates a clean file for downstream VEP peak detection or waveform analysis.

Use this file as the input to the later analysis notebook:

```text
outputs/preprocessed_waveforms/retained_cleaned_waveforms.csv
```


In [ ]:
if not filtered_df.empty and not summary_df.empty:
    retained_files = summary_df.loc[summary_df["rejected"] == False, "file"].tolist()
    retained_cleaned_df = filtered_df[filtered_df["file"].isin(retained_files)].copy()

    retained_cleaned_df.to_csv(OUTPUT_DIR / "retained_cleaned_waveforms.csv", index=False)

    print("Saved:", OUTPUT_DIR / "retained_cleaned_waveforms.csv")
    print("Retained traces:", len(retained_cleaned_df))


## 10. Notes on choosing filter settings

Recommended starting settings for noisy electrophysiology/VEP-like waveforms:

- baseline: first 25–50 samples if pre-stimulus is included
- median filter: good for single-sample spikes
- Savitzky-Golay: good for smoothing while preserving waveform shape
- low-pass Butterworth: useful only if sampling rate and frequency content are known

Avoid over-filtering.  
Over-filtering can shift peak timing or reduce true amplitudes.

For downstream VEP analysis, check:

- does N1 remain obvious?
- do P1/P2 remain visible?
- does the filtered trace preserve latency?
- are bad traces actually rejected?
- are real traces not being rejected?
